# Site search with Webflow and Shaped

This notebook demonstrates how to prepare, store, and retrieve documents from a Shaped relevance engine. We will use a real-world example of the Shaped Webflow site as an example. We will cover the following steps: 
1. Setup: Install dependencies and set Shaped API key
2. Ingestion: Chunking and inserting documents into a Shaped custom dataset
3. Inference: Making a text query to the Shaped relevance engine and retrieving results

# 1. Setup

First, we'll install the needed libraries and set our API keys.

In [1]:
%pip install -qU shaped webflow langchain-text-splitters lxml


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Initialize Shaped CLI with your API key

We will need to get a [free Shaped API key with write permissions](https://docs.shaped.ai/docs/support/getting-an-api-key). We'll then attach this to the Shaped Python SDK:

In [2]:
import os
from getpass import getpass

if (os.getenv("SHAPED_API_KEY") is None):
    SHAPED_API_KEY = getpass("Please enter your Shaped API key: ")
    os.environ["SHAPED_API_KEY"] = SHAPED_API_KEY

if (os.getenv("WEBFLOW_API_KEY") is None):
    WEBFLOW_API_KEY = getpass("Please enter your Webflow API key: ")
    os.environ["WEBFLOW_API_KEY"] = WEBFLOW_API_KEY

if (os.getenv("WEBFLOW_SITE_ID") is None):
    WEBFLOW_SITE_ID = getpass("Please enter your Webflow Site ID: ")
    os.environ["WEBFLOW_SITE_ID"] = WEBFLOW_SITE_ID


## 2. Ingestion

Next, we'll declare a new table to store our documents and insert rows to it. We'll cover the following steps in this section: 
1. Create a table with the Shaped API
2. Get documents with the Webflow API
3. Run the documents through a chunker
4. Upload the document chunks to our Shaped table

In [3]:
import json
import select
from webflow.client import Webflow
from getpass import getpass
import pandas as pd
from datetime import datetime

# Webflow bug workaround - 
# Monkey patch CollectionItemFieldData to support extra config (necessary to
# retrieve pydantic.Extra.allow reference from same pydantic reference used by
# webflow)
try:
    import pydantic.v1 as pydantic
except ImportError:
    import pydantic
from webflow import CollectionItemFieldData
CollectionItemFieldData.Config.extra = pydantic.Extra.allow

# declare new table with Shaped API

# TODO: Move the webflow bits to its own notebook (not super relevant for site search)

if (os.getenv("WEBFLOW_COLLECTION_ID") is None):
    COLLECTION_ID = getpass("Enter the Webflow collection ID for your blog: ")
    os.environ["WEBFLOW_COLLECTION_ID"] = COLLECTION_ID

webflowClient = Webflow(access_token=os.getenv("WEBFLOW_API_KEY"))

items = []
limit = 100
offset = 0

while True:
    items_page = webflowClient.collections.items.list_items(
        collection_id=COLLECTION_ID,
        limit=limit,
        offset=offset,
    )
    
    if items_page.items:
        items.extend(items_page.items)
        print(f"Fetched {len(items_page.items)} items (total: {len(items)})")
    else:
        break
    
    if len(items_page.items) < limit:
        break
    offset += limit

blog_posts_df = pd.DataFrame()

for item in items:
    item_id = item.id
    
    # Create a row dictionary starting with the item id
    row = {"id": item.id}
    
    # Destructure field_data into separate columns
    if hasattr(item, "field_data") and item.field_data:
        # Convert field_data to dict to get all fields
        field_data_dict = item.field_data.dict() if hasattr(item.field_data, "dict") else {}
        # Merge field_data fields into the row
        row.update(field_data_dict)

    now_str = datetime.now().isoformat()
    row["created-at"] = now_str
    row["updated-at"] = now_str
    
    # Append row to dataframe
    blog_posts_df = pd.concat([blog_posts_df, pd.DataFrame([row])], ignore_index=True)

blog_posts_df.to_json("posts.jsonl", orient="records", lines=True)
print("Blog posts have been successfully saved to posts.jsonl.")

# page = webflowClient.collections.items.get_item(collection_id=COLLECTION_ID, item_id=item_id)

# Run item through chunker - 
# https://api.python.langchain.com/en/latest/text_splitters/base/langchain_text_splitters.base.TextSplitter.html#langchain_text_splitters.base.TextSplitter

# 

Fetched 100 items (total: 100)
Fetched 100 items (total: 200)
Fetched 80 items (total: 280)
Blog posts have been successfully saved to posts.jsonl.


## Chunking

Now that we have our table of posts, we should extract sections to searches more relevant. To do this, we use a `chunking strategy`. 

In [9]:
from langchain_text_splitters import HTMLHeaderTextSplitter,HTMLSectionSplitter

# chunking step
blog_posts_chunked_df = pd.DataFrame()

# import posts from JSONL file
posts = pd.read_json('posts.jsonl', lines=True)
print(posts.columns)

post_to_chunk = posts.iloc[0]['post-body']

headers_to_split_on = [
    ("h1", "Header 1"),
    ("h2", "Header 2"),
    ("h3", "Header 3")
]

html_section_splitter = HTMLSectionSplitter(headers_to_split_on)

transformed = html_section_splitter.convert_possible_tags_to_header(post_to_chunk)

print(transformed)


Index(['id', 'name', 'slug', 'categories', 'popular', 'release-date',
       'post-body', 'author', 'roles', 'main-image', 'read-length-in-mins',
       'featured', 'post-summary', 'created-at', 'updated-at'],
      dtype='object')
<html><body>
<p id="">But here’s the thing: these same laws hold in domains other than language. In fact, some of the most consequential applications of scaling laws today are invisible to the end-user. They’re running under the hood of your credit card payments, your Netflix home screen, and your ride-share app’s matching system.</p>
<p id="">And unlike LLMs, the interface is not text – it’s embeddings.</p>
<figure id="" class="w-richtext-figure-type-image w-richtext-align-fullwidth" style="max-width:1470px" data-rt-type="image" data-rt-align="fullwidth" data-rt-max-width="1470px"><div id=""><img src="https://cdn.prod.website-files.com/6696d42284cfe85e5e20165b/6916320b63e4191c1c59894e_scaling-laws-shaped-2020.png" loading="lazy" alt="__wf_reserved_inherit" 

In [12]:
import re

def clean_invisible_characters(text):
    """Remove zero-width joiners and other invisible Unicode characters."""
    # Remove zero-width joiners, non-joiners, spaces, and formatting marks
    return re.sub(r'[\u200B-\u200D\uFEFF\u200E\u200F\u202A-\u202E]', '', text)

html_header_splitter = HTMLHeaderTextSplitter(headers_to_split_on)

# Clean the HTML content before chunking
post_to_chunk_cleaned = clean_invisible_characters(post_to_chunk)

print(post_to_chunk)

chunks = html_header_splitter.split_text(post_to_chunk)
print(len(chunks))
for chunk in chunks:
    print("meta - ", chunk.metadata)
    print(chunk.page_content)
    print()

<p id="">But here’s the thing: these same laws hold in domains other than language. In fact, some of the most consequential applications of scaling laws today are invisible to the end-user. They’re running under the hood of your credit card payments, your Netflix home screen, and your ride-share app’s matching system.</p><p id="">And unlike LLMs, the interface is not text – it’s embeddings.</p><figure id="" class="w-richtext-figure-type-image w-richtext-align-fullwidth" style="max-width:1470px" data-rt-type="image" data-rt-align="fullwidth" data-rt-max-width="1470px"><div id=""><img src="https://cdn.prod.website-files.com/6696d42284cfe85e5e20165b/6916320b63e4191c1c59894e_scaling-laws-shaped-2020.png" loading="lazy" alt="__wf_reserved_inherit" width="auto" height="auto" id=""></div><figcaption id=""><em id="">Scaling Laws for Neural Language Models (https://arxiv.org/pdf/2001.08361, 2020)</em></figcaption></figure><h2 id=""><strong id="">1. The Three Eras of Machine Learning</strong></h

In [ ]:
from langchain_text_splitters import HTMLSemanticPreservingSplitter

splitter = HTMLSemanticPreservingSplitter(
    headers_to_split_on=headers_to_split_on,
    separators=["\n\n", "\n", ". ", "! ", "? "],
    max_chunk_size=50,
    elements_to_preserve=["ul", "ol", "code"],
    denylist_tags=["script", "style", "head"],
)
documents = splitter.split_text(post_to_chunk)
documents

/var/folders/p9/1p33b6wd57n39qdrj2lh_t5r0000gn/T/ipykernel_62672/868317147.py:3: LangChainBetaWarning: The class `HTMLSemanticPreservingSplitter` is in beta. It is actively being worked on, so the API may change.
  splitter = HTMLSemanticPreservingSplitter(


[Document(metadata={}, page_content='But here’s the thing: these same laws hold in domains other than language'),
 Document(metadata={}, page_content='. In fact, some of the most consequential applications of scaling laws today are invisible to the end-user'),
 Document(metadata={}, page_content='. They’re running under the hood of your credit card payments, your Netflix home screen, and your ride-share app’s matching system'),
 Document(metadata={}, page_content='. And unlike LLMs, the interface is not text – it’s embeddings'),
 Document(metadata={}, page_content='. Scaling Laws for Neural Language Models (https://arxiv.org/pdf/2001.08361, 2020)'),
 Document(metadata={'Header 2': '1. The Three Eras of Machine Learning'}, page_content="To fully grasp this shift, it's helpful to see it as the third major era of machine learning: ML 0.0: Pre-Deep Learning. This was the era of hand-crafted features, logistic regression, SVMs, and gradient boosting. Systems were powerful but brittle, requi